# Loan Default Prediction — Modelling

This notebook builds and evaluates three classification models in order of 
complexity: Logistic Regression (baseline), Random Forest, and XGBoost.
Processed data is loaded from `data/processed/train_test_data.pkl` produced 
by `02_preprocessing.ipynb`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)

# Load preprocessed data
with open('../data/processed/train_test_data.pkl', 'rb') as f:
    X_train, X_test, y_train, y_test = pickle.load(f)
One minor imprecision worth knowing about — the line "ROC-AUC is unaffected by class imbalance" is the standard framing and is mostly correct, but a pedantic statistician would say it's less sensitive to imbalance rather than completely unaffected. This is unlikely to be flagged by a recruiter, and Day 8's Precision-Recall curve will add the more imbalance-aware metric anyway, so you're covered.
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train default rate: {y_train.mean():.4f}")
print(f"y_test  default rate: {y_test.mean():.4f}")

SyntaxError: invalid character '—' (U+2014) (3220358910.py, line 17)

## Model 1: Logistic Regression (Baseline)

Logistic Regression is a Generalised Linear Model (GLM) with a logit link 
function. Rather than minimising a sum of squared errors like OLS regression, 
it finds coefficients by **maximising the log-likelihood** of the observed 
binary outcomes, which is a direct application of Maximum Likelihood Estimation (MLE).

The model estimates the probability that a loan defaults:

    P(Default=1 | X) = 1 / (1 + exp(−Xβ))

where β are the coefficients estimated via MLE.

**Why `class_weight='balanced'`:** The training set has an 88.4/11.6 class 
split. Without adjustment, the model is incentivised to predict "no default" 
for everything and still achieve high accuracy. Setting `class_weight='balanced'` 
instructs sklearn to weight each class inversely proportional to its frequency, 
effectively penalising misclassification of the minority class (defaulters) more 
heavily. This is more principled than resampling methods like SMOTE because it 
operates on the loss function directly rather than artificially duplicating data.

**Why `max_iter=1000`:** The default of 100 iterations is sometimes insufficient 
for convergence on larger datasets. 1000 ensures the solver reaches the optimum.

In [ ]:
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)
print("Logistic Regression training complete.")

In [ ]:
# Predicted class labels (0 or 1)
lr_pred = lr_model.predict(X_test)

# Predicted probabilities of default (class 1)
# We need probabilities, not just labels, to compute ROC-AUC
lr_prob = lr_model.predict_proba(X_test)[:, 1]

## Evaluation — Logistic Regression

We evaluate using three complementary tools:

**1. ROC-AUC:** Measures how well the model separates defaulters from 
non-defaulters across all possible thresholds. AUC = 1.0 is perfect; 
AUC = 0.5 is random. This is our primary metric because it is unaffected 
by class imbalance.

The ROC curve directly visualises the Neyman-Pearson trade-off: at every 
threshold, we trade True Positive Rate (sensitivity / statistical power) 
against False Positive Rate (Type I error rate). A model pushed toward high 
sensitivity catches more defaulters but flags more good applicants as risky.

**2. Confusion Matrix:** Shows the four outcomes at the default 0.5 threshold:
- True Negatives (correctly predicted no default)
- False Positives (predicted default, actually fine — Type I error)  
- False Negatives (predicted no default, actually defaulted — Type II error)
- True Positives (correctly predicted default)

In a lending context, **False Negatives are more costly** — approving a loan 
that defaults causes direct financial loss. False Positives lose potential 
revenue but cause no loss.

**3. Classification Report:** Precision, Recall, and F1-score per class.

In [ ]:
lr_auc = roc_auc_score(y_test, lr_prob)
print(f"Logistic Regression ROC-AUC: {lr_auc:.4f}")

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, lr_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {lr_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random baseline (AUC = 0.5)')
plt.xlabel('False Positive Rate (Type I Error Rate)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve — Logistic Regression')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(y_test, lr_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, 
                               display_labels=['No Default', 'Default'])
disp.plot(cmap='Blues')
for text in disp.text_.ravel():
    text.set_text(f'{int(float(text.get_text())):,}')
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

In [ ]:
print("Classification Report — Logistic Regression")
print("=" * 50)
print(classification_report(y_test, lr_pred, 
                             target_names=['No Default', 'Default']))

### Coefficient Interpretation

A key advantage of Logistic Regression over tree-based models is 
interpretability. Each coefficient β can be exponentiated to give an 
**odds ratio**: exp(β).

An odds ratio > 1 means that feature increases the odds of default; 
< 1 means it decreases the odds. This connects directly to how risk is 
communicated in real financial settings.

In [ ]:
feature_names = X_train.columns.tolist()
coefficients = lr_model.coef_[0]

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Odds Ratio': np.exp(coefficients)
}).sort_values('Coefficient', ascending=False)

print(coef_df.to_string(index=False))

### Logistic Regression — Results Interpretation

The Logistic Regression model achieved an **ROC-AUC score of 0.7532**, which means it can distinguish between defaulters and non-defaulters reasonably well. Since a score of 0.5 represents random guessing, this serves as a solid baseline for comparing more advanced models.

**Confusion Matrix (Threshold = 0.5):**
- **4,148 True Positives:** Defaulters correctly identified.
- **1,783 False Negatives:** Defaulters who were mistakenly predicted as safe borrowers.
- **14,749 False Positives:** Creditworthy applicants incorrectly classified as risky.
- **~30,000 True Negatives:** Non-defaulters correctly classified.

The model produces a relatively high number of false positives because `class_weight='balanced'` gives extra importance to detecting the minority class (defaulters). As a result, it achieves a good recall (0.70) but lower precision (0.22). In loan approval problems, this trade-off is often acceptable since approving a borrower who later defaults is generally more costly than rejecting someone who would have repaid the loan.

### Most Important Risk Factors

The variables with the strongest positive impact on the likelihood of default are:

- **Interest Rate (OR = 1.58):** The strongest predictor in the model. Borrowers with higher interest rates are more likely to default, which is consistent with the findings from the exploratory data analysis.
- **Employment Type – Unemployed (OR = 1.55):** Unemployed borrowers have about **55% higher odds** of default compared to full-time employees.
- **Loan Amount (OR = 1.33):** Larger loan amounts are associated with a higher risk of default.

### Factors Associated with Lower Default Risk

The variables that reduce the likelihood of default are:

- **Age (OR = 0.56):** Older borrowers are less likely to default.
- **Months Employed (OR = 0.71):** A longer employment history is associated with lower default risk.
- **Income (OR = 0.73):** Higher-income borrowers are generally more reliable in repaying loans.

Overall, the model's results closely match the patterns observed during the exploratory data analysis, suggesting that it has learned meaningful relationships from the data rather than random patterns.

*Next, we evaluate the Random Forest model, which can capture more complex, non-linear relationships between the features and the target variable.*

## Model 2: Random Forest

Random Forest is an ensemble learning algorithm that combines the predictions of many decision trees instead of relying on a single tree. Each tree is trained on a different random sample of the data, and the final prediction is made through majority voting. This approach helps the model capture **non-linear relationships** and **complex interactions between features** that a Logistic Regression model may miss.

The following hyperparameters were used:

- **n_estimators = 100:** The model builds 100 decision trees. This is a commonly used value that provides stable performance without requiring excessive computation.
- **class_weight = 'balanced':** Since the dataset is imbalanced (around 88% non-defaulters and 12% defaulters), balanced class weights give more importance to the minority class during training. This helps the model identify defaulters more effectively instead of favoring the majority class.
- **random_state = 42:** Ensures that the results are reproducible each time the model is trained.
- **n_jobs = -1:** Uses all available CPU cores to train the trees in parallel, reducing training time.

Unlike Logistic Regression, Random Forest does not require feature scaling because decision trees make splits based on feature values rather than distances. Therefore, applying the `StandardScaler` has no impact on the model's performance, although it also does not cause any issues.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print("Random Forest training complete.")

In [ ]:
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

### Evaluation — Random Forest

We apply the same three-tool evaluation framework as Logistic Regression:
ROC-AUC as the primary metric, confusion matrix for Type I and Type II error inspection,
and the classification report for precision, recall, and F1.

In [ ]:
rf_auc = roc_auc_score(y_test, rf_prob)
print(f"Random Forest ROC-AUC: {rf_auc:.4f}")

fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {rf_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random baseline (AUC = 0.5)')
plt.xlabel('False Positive Rate (Type I Error Rate)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve — Random Forest')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
cm_rf = confusion_matrix(y_test, rf_pred)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf,
                                  display_labels=['No Default', 'Default'])
disp_rf.plot(cmap='Blues')
for text in disp_rf.text_.ravel():
    text.set_text(f'{int(float(text.get_text())):,}')
plt.title('Confusion Matrix — Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
print("Classification Report — Random Forest")
print("=" * 50)
print(classification_report(y_test, rf_pred,
                             target_names=['No Default', 'Default']))

In [ ]:
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importances — Random Forest (Gini Impurity)')
plt.xlabel('Mean Decrease in Gini Impurity')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

### Random Forest — Results Interpretation

The Random Forest model achieved an **ROC-AUC score of 0.7411**, which is slightly lower than the Logistic Regression model (0.7532). Although Random Forest is often expected to outperform linear models, that was not the case here. A likely reason is that this dataset is synthetically generated and does not contain strong non-linear patterns for the model to take advantage of.

**Confusion Matrix (Threshold = 0.5):**
- **940 True Positives:** Defaulters correctly identified.
- **4,991 False Negatives:** Defaulters incorrectly classified as non-defaulters.
- **1,169 False Positives:** Creditworthy applicants incorrectly classified as risky.
- **43,970 True Negatives:** Non-defaulters correctly identified.

Compared to Logistic Regression, Random Forest is much more conservative when predicting defaults. While it achieved a higher overall accuracy (0.88), its recall for the default class dropped from **0.70** to **0.16**, meaning it failed to identify many borrowers who actually defaulted. Even with `class_weight='balanced'`, the model still favored the majority class more than Logistic Regression did.

### Feature Importance

The feature importance scores show that **Interest Rate, Age, Income, Loan Amount, Months Employed, Credit Score,** and **DTI Ratio** were the most influential variables. Together, these continuous features contributed nearly **80%** of the total importance.

In contrast, the one-hot encoded categorical variables such as **Employment Type, Loan Purpose, Education,** and **Marital Status** had much smaller and fairly similar importance values. This agrees with the Logistic Regression results, where the same continuous variables had the strongest influence on predicting loan default.

It is important to note that feature importance in Random Forest only tells us **how useful a feature was for making splits** across the trees. Unlike the coefficients in Logistic Regression, it does not indicate whether a feature increases or decreases the risk of default. For understanding the direction of a feature's effect, the Logistic Regression odds ratios provide a clearer interpretation.

## Baseline Comparison: Logistic Regression vs Random Forest

With both models trained and evaluated independently, we now compare them directly on the
same test set. The combined ROC curve shows which model achieves a better
sensitivity-specificity trade-off across all decision thresholds simultaneously.

In [ ]:
# Recompute LR curve with explicit naming for the comparison plot
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_prob)

plt.figure(figsize=(9, 7))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {lr_auc:.4f})', linewidth=2)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest       (AUC = {rf_auc:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random baseline (AUC = 0.5)', linewidth=1)
plt.xlabel('False Positive Rate (Type I Error Rate)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curve Comparison — Logistic Regression vs Random Forest', fontsize=14)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'ROC-AUC': [round(lr_auc, 4), round(rf_auc, 4)]
}).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

comparison_df.index += 1
print("Performance Comparison: Logistic Regression vs Random Forest")
print("=" * 55)
print(comparison_df.to_string())

### Logistic Regression vs Random Forest — Summary

| Model | ROC-AUC | Default Recall | Default Precision |
|---|---:|---:|---:|
| Logistic Regression | 0.7532 | 0.70 | 0.22 |
| Random Forest | 0.7411 | 0.16 | 0.45 |

Overall, **Logistic Regression performed better than Random Forest** on this dataset. It achieved a slightly higher ROC-AUC score and, more importantly, identified a much larger proportion of actual defaulters (recall = 0.70 compared to 0.16).

Although Random Forest had higher precision (0.45), it missed many borrowers who eventually defaulted. In a loan approval setting, failing to identify defaulters is generally more costly than incorrectly flagging some creditworthy applicants, making Logistic Regression the more suitable model in this case.

The lower performance of Random Forest does not necessarily mean that it is a weaker algorithm. Since this is a synthetic dataset, the relationships between the features and the target appear to be mostly linear. As a result, Logistic Regression is able to model these patterns effectively, leaving little advantage for a more complex model like Random Forest.

The next model evaluated is **XGBoost**, a gradient boosting algorithm that builds trees sequentially by correcting the errors made by previous trees. It may be able to capture patterns that both Logistic Regression and Random Forest fail to learn.

## Model 3: XGBoost

XGBoost (Extreme Gradient Boosting) is a sequential ensemble method. Unlike Random Forest,
which builds trees independently in parallel and averages them, XGBoost builds each new tree
to **correct the residual errors** of all previous trees combined. This iterative error-correction
strategy is called gradient boosting and often produces stronger predictive performance on
structured tabular data.

Key hyperparameters:

- **n_estimators=100**: 100 boosting rounds. Each round adds one tree that targets the
  errors the current ensemble is still making.
- **learning_rate=0.1**: Controls how much each tree's correction contributes to the final
  prediction. Lower values are more conservative and reduce overfitting; 0.1 is a standard
  starting point.
- **scale_pos_weight**: XGBoost's built-in mechanism for handling class imbalance. Set to
  the ratio of negative to positive samples in the training set, which instructs XGBoost to
  penalise misclassification of the minority class (defaulters) proportionally more. This is
  computed directly from the training data rather than chosen arbitrarily.
- **eval_metric='logloss'**: Suppresses a deprecation warning and explicitly sets the
  evaluation metric used internally during training.
- **random_state=42**: Reproducibility.
- **n_jobs=-1**: Parallelises computation across all CPU cores.

In [ ]:
from xgboost import XGBClassifier

# Compute scale_pos_weight from training labels
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
spw = neg_count / pos_count
print(f"scale_pos_weight = {spw:.2f}  (negatives: {neg_count:,} / positives: {pos_count:,})")

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
print("XGBoost training complete.")

In [ ]:
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

### Evaluation — XGBoost

We apply the same three-tool evaluation framework used for both previous models.

In [ ]:
xgb_auc = roc_auc_score(y_test, xgb_prob)
print(f"XGBoost ROC-AUC: {xgb_auc:.4f}")

fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {xgb_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random baseline (AUC = 0.5)')
plt.xlabel('False Positive Rate (Type I Error Rate)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve — XGBoost')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
cm_xgb = confusion_matrix(y_test, xgb_pred)
disp_xgb = ConfusionMatrixDisplay(confusion_matrix=cm_xgb,
                                   display_labels=['No Default', 'Default'])
disp_xgb.plot(cmap='Blues')
for text in disp_xgb.text_.ravel():
    text.set_text(f'{int(float(text.get_text())):,}')
plt.title('Confusion Matrix — XGBoost')
plt.tight_layout()
plt.show()

In [ ]:
print("Classification Report — XGBoost")
print("=" * 50)
print(classification_report(y_test, xgb_pred,
                             target_names=['No Default', 'Default']))

In [ ]:
xgb_importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(data=xgb_importance_df, x='Importance', y='Feature', palette='magma')
plt.title('Feature Importances — XGBoost (Gain)')
plt.xlabel('Feature Importance Score (Gain)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print(xgb_importance_df.to_string(index=False))

### XGBoost — Results Interpretation

The XGBoost model achieved an **ROC-AUC score of 0.7552**, making it the best-performing model among the three. Although the improvement over Logistic Regression (0.7532) is small, it suggests that XGBoost was able to capture a few additional patterns in the data that the other models could not.

**Confusion Matrix (Threshold = 0.5):**
- **3,964 True Positives:** Defaulters correctly identified.
- **1,967 False Negatives:** Defaulters incorrectly classified as non-defaulters.
- **13,330 False Positives:** Creditworthy applicants incorrectly classified as risky.
- **31,809 True Negatives:** Non-defaulters correctly identified.

The model achieved a **default recall of 0.67**, which is very close to Logistic Regression (0.70) and much higher than Random Forest (0.16). This means XGBoost was able to identify most defaulters while also achieving the highest overall ROC-AUC score, making it the strongest performer in this comparison.

### Feature Importance

XGBoost measures feature importance using **gain**, which represents the average improvement in the model's objective function whenever a feature is used to split the data. Features with higher gain contribute more to improving the model's predictions.

The most important feature was **Age**, followed by **Interest Rate** and **Income**. While these variables were also among the most important in the Random Forest model, XGBoost placed much greater importance on Age.

Interestingly, **HasCoSigner** and **HasDependents** received higher importance scores in XGBoost than they did in Random Forest. This suggests that these variables helped improve predictions during the boosting process, even though they were not selected as frequently by Random Forest.

Like Random Forest, XGBoost's feature importance scores only indicate **how important** a feature is for prediction. They do not show whether a feature increases or decreases the likelihood of default. For understanding the direction of each feature's effect, the coefficients and odds ratios from the Logistic Regression model remain easier to interpret.

## Final Model Comparison

All three models have been trained and evaluated on the same held-out test set. We now
compare them directly to identify which achieves the best separation between defaulters and
non-defaulters across all decision thresholds.

In [ ]:
plt.figure(figsize=(9, 7))
plt.plot(fpr_lr,  tpr_lr,  label=f'Logistic Regression (AUC = {lr_auc:.4f})', linewidth=2)
plt.plot(fpr_rf,  tpr_rf,  label=f'Random Forest       (AUC = {rf_auc:.4f})', linewidth=2)
plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost             (AUC = {xgb_auc:.4f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random baseline (AUC = 0.5)', linewidth=1)
plt.xlabel('False Positive Rate (Type I Error Rate)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curve Comparison — All Models', fontsize=14)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [2]:
final_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'ROC-AUC': [round(lr_auc, 4), round(rf_auc, 4), round(xgb_auc, 4)]
}).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

final_comparison.index += 1
print("Final Model Performance Summary")
print("=" * 40)
print(final_comparison.to_string())

NameError: name 'pd' is not defined

## Final Model Recommendation

Among the three models, **XGBoost** achieved the highest **ROC-AUC score (0.7552)**, making it the best-performing model for predicting loan defaults in this project.

| Model | ROC-AUC | Default Recall | Default Precision |
|---|---:|---:|---:|
| XGBoost | 0.7552 | 0.67 | 0.23 |
| Logistic Regression | 0.7532 | 0.70 | 0.22 |
| Random Forest | 0.7411 | 0.16 | 0.45 |

### Key Findings

1. **XGBoost** delivered the best overall performance with the highest ROC-AUC score while maintaining a high recall (0.67). It successfully balanced identifying defaulters with overall predictive performance, making it the recommended model for this dataset.

2. **Logistic Regression** performed almost as well as XGBoost. It achieved a slightly higher recall (0.70), meaning it identified a few more actual defaulters. In addition, its coefficients and odds ratios make the model easy to interpret, which is an important advantage in applications such as credit risk assessment where model decisions need to be explained.

3. **Random Forest** performed worse than the other two models, mainly because it identified relatively few defaulters (recall = 0.16). Although it achieved good overall accuracy, it was less effective at detecting the minority class, making it less suitable for this task.

Overall, the differences in performance between XGBoost and Logistic Regression were quite small. This suggests that the relationships in the dataset are mostly linear, allowing a simple linear model to perform nearly as well as more complex ensemble methods.

It is also worth noting that this dataset is synthetically generated. On real-world loan datasets, which often contain more complex relationships, missing values, and stronger interactions between variables, models such as XGBoost generally have a greater advantage over simpler models like Logistic Regression.